# Scheme: how bufferToWorker serves the plates

The mini-job's blocks sit on the buffet (left), cut into plates of 4 blocks. Each worker (a colour; its hexagon shows the plate it is decoding) takes the next serving from the shared counter, decodes that plate, and writes it at the same place in the output (right), where each block takes 8 cells (its f32 values). Worker 1 took plates 0 and 3; plate 4 is the next serving.

In [ ]:
import re
import subprocess
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# the notebook sits in Claude Benchmark/<name>/, the repository two levels up
HERE = Path.cwd().resolve()
ROOT = HERE.parents[1]

MM = 1 / 25.4
INK, PAPER, GREY, SHADE = "#1f1d1a", "#faf8f3", "#77746e", "#d3cfc6"
BASELINE_GREY = "#a3a09a"
plt.rcParams.update({
    "font.family": "Liberation Sans", "font.size": 7, "axes.labelsize": 7,
    "xtick.labelsize": 6.5, "ytick.labelsize": 6.5, "legend.fontsize": 6.5,
    "axes.edgecolor": INK, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK, "text.color": INK,
    "axes.linewidth": 0.6, "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "figure.facecolor": PAPER, "axes.facecolor": PAPER, "savefig.facecolor": PAPER,
    "axes.spines.top": False, "axes.spines.right": False,
})


def main_constant(name):
    """A constant of src/main.zig, such as `const MIN_BLOCKS_FOR_WORKERS = 16 * 1024;`, evaluated."""
    source = (ROOT / "src" / "main.zig").read_text(encoding="utf-8")
    expression = re.search(rf"^const {name} = ([0-9 *]+);", source, re.MULTILINE).group(1)
    return int(np.prod([int(factor) for factor in expression.split("*")]))


def run_benchmark(bench, csv):
    """Runs the benchmark (calls of bufferToWorker only) when its results are missing."""
    if not (HERE / csv).exists():
        subprocess.run(["zig", "run", "-OReleaseFast", "-mcpu=native", "--dep", "dequantizer",
                        f"-Mroot={HERE / bench}", f"-Mdequantizer={ROOT / 'src' / 'root.zig'}"],
                       cwd=ROOT, check=True)
    return pd.read_csv(HERE / csv)


In [ ]:
from matplotlib.colors import to_rgb
from matplotlib.patches import FancyArrowPatch, Rectangle, RegularPolygon

blocks_per_plate, plates = 4, 5
taken = {0: 0, 1: 1, 2: 2, 3: 0}        # plate -> worker that took it, plate 4 is still on the buffet
next_serving = 4
# ink blue, brick, ochre: muted, and still distinct under colour-blind simulation (worst pair dE 27)
worker_colour = ["#34558b", "#b5533c", "#d1a042"]
LINE = "#c9c3b8"                 # thin lines of the empty cells

def tint(colour, amount):
    # the colour mixed into the paper
    paper = np.array(to_rgb(PAPER))
    return tuple(paper + amount * (np.array(to_rgb(colour)) - paper))

fig, ax = plt.subplots(figsize=(165 * MM, 100 * MM))
ax.set_xlim(0.2, 13.85)
ax.set_ylim(-0.1, 10.2)
ax.set_aspect("equal")
ax.axis("off")

top, bottom_edge = 10.0, 0.9
gap = 0.22                       # air between plates
plate_height = (top - bottom_edge - gap * (plates - 1)) / plates
block_height = plate_height / blocks_per_plate
cell = 0.55                      # one packed block on the buffet
buffet_x = 1.0
worker_x = 5.3
output_x = 8.6                   # 8 cells per block in the output: its f32 values take 8 times the room
output_width = 8 * cell
label_y = 0.45

def plate_bottom(plate):
    return top - (plate + 1) * plate_height - plate * gap

def cells(x, y, columns, fill):
    for column in range(columns):
        ax.add_patch(Rectangle((x + column * cell, y), cell, block_height, facecolor=fill, edgecolor=LINE, lw=0.6))

for plate in range(plates):
    bottom = plate_bottom(plate)
    middle = bottom + plate_height / 2
    served = plate in taken
    colour = worker_colour[taken[plate]] if served else None
    for block in range(blocks_per_plate):
        y = bottom + (blocks_per_plate - 1 - block) * block_height
        # a served plate is lightly tinted on the buffet and filled in the output
        cells(buffet_x, y, 1, tint(colour, 0.18) if served else PAPER)
        cells(output_x, y, 8, tint(colour, 0.38) if served else PAPER)
    ax.text(buffet_x - 0.3, middle, str(plate), ha="right", va="center", fontsize=7, color=GREY)
    if served:
        for x, width in ((buffet_x, cell), (output_x, output_width)):
            ax.add_patch(Rectangle((x, bottom), width, plate_height, fill=False, edgecolor=colour, lw=1.1))
        ax.add_patch(FancyArrowPatch((buffet_x + cell + 0.2, middle), (worker_x - 0.62, middle),
                                     arrowstyle="-|>", mutation_scale=7, color=colour, lw=0.9))
        ax.add_patch(FancyArrowPatch((worker_x + 0.62, middle), (output_x - 0.2, middle),
                                     arrowstyle="-|>", mutation_scale=7, color=colour, lw=0.9))
        ax.add_patch(RegularPolygon((worker_x, middle), numVertices=6, radius=0.5, orientation=np.pi / 6,
                                    facecolor=tint(colour, 0.22), edgecolor=colour, lw=1.1, zorder=4))
        # the colour says which worker, the number which plate it is decoding
        ax.text(worker_x, middle, str(plate), ha="center", va="center", fontsize=8, fontweight="bold",
                color=colour, zorder=5)
    else:
        for x, width in ((buffet_x, cell), (output_x, output_width)):
            ax.add_patch(Rectangle((x, bottom), width, plate_height, fill=False, edgecolor=GREY, lw=0.9,
                                   ls=(0, (3, 2))))

# the counter the workers share, on the row of the plate it serves next
counter_y = plate_bottom(next_serving) + plate_height / 2
ax.plot([worker_x, worker_x], [counter_y + 0.42, plate_bottom(0) + plate_height / 2], color=GREY, lw=0.7,
        ls=(0, (1.5, 2.5)), zorder=0)
ax.add_patch(Rectangle((worker_x - 0.55, counter_y - 0.38), 1.1, 0.76, facecolor=PAPER, edgecolor=INK, lw=0.9, zorder=4))
ax.text(worker_x, counter_y, str(next_serving), ha="center", va="center", fontsize=11, color=INK, zorder=5)
ax.add_patch(FancyArrowPatch((worker_x - 0.65, counter_y), (buffet_x + cell + 0.2, counter_y),
                             arrowstyle="-|>", mutation_scale=7, color=GREY, lw=0.8, ls=(0, (3, 2))))
ax.text(worker_x, counter_y - 0.52, "next serving", ha="center", va="top", fontsize=6.5, color=GREY, style="italic")

# the three names on one line
for x, name in ((buffet_x + cell / 2, "buffet"), (worker_x, "workers"), (output_x + output_width / 2, "output (f32)")):
    ax.text(x, label_y, name, ha="center", va="top", fontsize=7, color=INK)
fig.savefig(HERE / "scheme.png", dpi=300, bbox_inches="tight", pad_inches=0.25)